# S&P 500 Daily Stock Price EDA

## Research question

> **How do recent return momentum and volatility vary across S&P 500 stocks, and are they associated with next-day return direction?**

This notebook explores the Kaggle dataset **S&P 500 Stocks: 25 Years of Data (Updated Daily)**. It uses daily price and volume history with company sector metadata. It creates lagged momentum, realized volatility, and volume features for the following baseline/ML notebook.

Run the preprocessing notebook first. The Kaggle source is described as refreshed daily; the universe contains current constituents, so historical cross-sectional patterns have survivorship bias. This data contains no EBITDA or market capitalization fields.

## 1. Load the prepared prices

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR=Path('/content/drive/MyDrive/sp500_kaggle_processed')
PRICE_PATH=DATA_DIR/'sp500_prices.parquet'
assert PRICE_PATH.exists(),'Run Quantitative_Finance_Stock_Market_Dataset_Project.ipynb first.'
prices=pd.read_parquet(PRICE_PATH)
prices['date']=pd.to_datetime(prices['date'])
prices=prices.sort_values(['symbol','date']).reset_index(drop=True)
use_adjusted=prices['adj_close'].notna().all()
price_col='adj_close' if use_adjusted else 'close'
prices['price_for_returns']=prices[price_col]
print('Observations:',len(prices),'| symbols:',prices['symbol'].nunique(),'| date range:',prices['date'].min().date(),'to',prices['date'].max().date())
print('Return price basis:',price_col)
display(prices.head())

### Why check the date range and price basis first

The last date should reflect the Kaggle version downloaded in the preprocessing notebook. The return-price field uses adjusted close only if that column is complete; otherwise it uses close. This makes the basis explicit before comparing returns and volatility.

## 2. Build lagged features and next-day direction label

All predictors use data available through the close on date *t*. The label is whether the next trading session's return is positive. These features are exported for chronological model training.

In [ ]:
g=prices.groupby('symbol',group_keys=False)
prices['daily_return']=g['price_for_returns'].pct_change()
prices['return_1d']=prices['daily_return']
prices['momentum_5d']=g['price_for_returns'].pct_change(5)
prices['momentum_20d']=g['price_for_returns'].pct_change(20)
prices['volatility_20d']=g['daily_return'].rolling(20,min_periods=20).std().reset_index(level=0,drop=True)
prices['log_volume']=np.log1p(prices['volume'].clip(lower=0))
prices['volume_change_5d']=prices.groupby('symbol')['log_volume'].diff(5)
prices['next_day_return']=prices.groupby('symbol')['daily_return'].shift(-1)
prices['target_date']=prices.groupby('symbol')['date'].shift(-1)
prices['next_day_up']=(prices['next_day_return']>0).astype('Int64')
FEATURES=['return_1d','momentum_5d','momentum_20d','volatility_20d','volume_change_5d','sector']
model_frame=prices.dropna(subset=['next_day_return','target_date',*FEATURES]).copy()
model_frame=model_frame.replace([np.inf,-np.inf],np.nan).dropna(subset=['next_day_up',*FEATURES])
print('Model rows:',len(model_frame),'| date range:',model_frame['date'].min().date(),'to',model_frame['date'].max().date())
print('Up-day share:',round(float(model_frame['next_day_up'].mean()),3))
display(model_frame[['date','target_date','symbol',*FEATURES,'next_day_return','next_day_up']].head())

### Reasoning behind the features and label

All features use information available through the current day's close. The label shifts the same symbol's return forward by one available trading observation. The first 20 days lack a full volatility window, and the final row for each symbol lacks a next-day label, so those rows are omitted from the modeling panel.

## 3. EDA graph 1 — equal-weighted sample cumulative return

This is the daily cross-sectional average return across available symbols, compounded through time. It is an equal-weighted sample portfolio proxy, not the official S&P 500 index.

In [ ]:
equal_weight_returns=model_frame.groupby('date')['daily_return'].mean().dropna()
equal_weight_growth=(1+equal_weight_returns).cumprod()
fig,ax=plt.subplots(figsize=(13,5))
ax.plot(equal_weight_growth.index,equal_weight_growth.values,color='#4472C4')
ax.set(title='Equal-weighted S&P 500 sample growth proxy',xlabel='Date',ylabel='Growth of $1 (daily average return)')
ax.grid(alpha=.25);fig.tight_layout();plt.show()

### How to interpret the cumulative-return chart

A rising line means the equal-weighted average of available stock returns compounded positively over that period. This is a sample portfolio proxy, not the official S&P 500 index; availability and sector representation can vary over time.

## 4. EDA graph 2 — rolling volatility for illustrative stocks

In [ ]:
sample_symbols=['AAPL','MSFT','NVDA','AMZN','GOOGL']
fig,ax=plt.subplots(figsize=(13,5))
for symbol in sample_symbols:
    series=prices.loc[prices['symbol'].eq(symbol)].sort_values('date').copy()
    if series.empty: continue
    rolling=series['daily_return'].rolling(20,min_periods=20).std()*np.sqrt(252)
    ax.plot(series['date'],rolling,label=symbol,linewidth=1)
ax.set(title='20-session annualized realized volatility',xlabel='Date',ylabel='Annualized volatility')
ax.grid(alpha=.25);ax.legend(ncol=5);fig.tight_layout();plt.show()

### How to interpret rolling volatility

The 20-session standard deviation is annualized by multiplying by the square root of 252. Peaks mark periods when a stock's recent daily returns varied more. The chart describes realized risk and does not forecast the next volatility level.

## 5. EDA graph 3 — momentum bins and next-day up frequency

The chart describes whether next-day direction frequency differs across 20-day momentum bins. It is descriptive and does not establish predictive power; the chronological validation comparison is in the next notebook.

In [ ]:
eda=model_frame[['momentum_20d','next_day_up']].dropna().copy()
eda['momentum_bin']=pd.qcut(eda['momentum_20d'],q=10,duplicates='drop')
up_rate=eda.groupby('momentum_bin',observed=True)['next_day_up'].mean()
fig,ax=plt.subplots(figsize=(11,5))
up_rate.plot(kind='bar',ax=ax,color='#70AD47')
ax.axhline(eda['next_day_up'].mean(),color='#C00000',linestyle='--',label='Overall up-day rate')
ax.set(title='Next-day positive-return frequency by 20-day momentum decile',xlabel='20-day momentum decile',ylabel='Next-day up frequency')
ax.tick_params(axis='x',rotation=35);ax.grid(axis='y',alpha=.25);ax.legend();fig.tight_layout();plt.show()

### How to interpret momentum bins

Each bar is the fraction of observations in a 20-day momentum decile that had a positive next-session return. The dashed line is the overall positive-return rate. Differences between bars are exploratory; they do not establish a reliable trading signal and should be checked on later dates.

## 6. Save the feature panel for baseline and ML modeling

The label uses the next trading session for the same symbol. Keep dates intact: the modeling notebook splits by calendar date so all symbols from a given date stay in the same partition.

In [ ]:
FEATURE_PATH=DATA_DIR/'sp500_next_day_features.parquet'
model_frame.to_parquet(FEATURE_PATH,index=False)
print('Saved',FEATURE_PATH)
print('Feature columns:',FEATURES)
print('Target: next_day_up (1 if next-session return > 0)')

### Why the saved panel is the model input

The Parquet panel keeps one row per symbol and feature date, with lagged predictors, the next-session target, and target date. The modeling notebook uses the target date to purge observations that cross chronological split boundaries.

## Interpretation limits

This dataset supplies daily market data and sector metadata, not SEC fundamentals. It cannot directly test EBITDA or market capitalization. The price history follows a currently represented S&P 500 universe, so backtests can omit companies removed from the index. Daily returns are noisy and the same market shocks affect many symbols on the same date.

# Detailed EDA guide: what each feature measures

## Return and momentum definitions

For symbol *i* on date *t*, let `P(i,t)` be the selected return price. The one-session simple return is

`r(i,t) = P(i,t) / P(i,t-1) - 1`.

The five-session momentum is `P(i,t) / P(i,t-5) - 1`; the 20-session momentum uses the same formula over 20 observed rows for that symbol. These are cumulative simple returns, not annualized returns. They summarize recent price direction and magnitude. A positive momentum value means the price is above its comparison point; it does not say that the next return will also be positive.

The 20-session realized volatility is the sample standard deviation of the previous 20 daily returns. In the rolling feature code, the window ends at the prediction date, so it contains no future returns. The volatility used as a feature is a risk summary, not the target. It can be high after large moves in either direction.

The five-session volume feature is the change in `log(1 + volume)`. The logarithm compresses the long right tail of volume, where a few unusually active sessions can dominate a raw scale. A positive value means log volume is higher than five observations earlier. It does not identify whether the activity came from buying, selling, or a news event.

The target `next_day_return` is the next recorded return for the same ticker. `next_day_up` is one when that return is greater than zero and zero otherwise. A zero return is grouped with down/flat in the displayed tree label; it is not a meaningful short-sale instruction.

## EDA graph 1: equal-weighted growth proxy

The chart first averages same-day returns across the available symbols, then compounds those daily averages. Each symbol receives equal weight for that day. This differs from an official S&P 500 index, which is market-cap weighted and uses index methodology, corporate actions, constituent changes, and divisor adjustments.

Use the graph to answer descriptive questions: Does the sample proxy rise or fall over the visible window? Are there sharp drawdowns? Does its endpoint move materially when the dataset is refreshed? Avoid turning a cumulative curve into a forecast. The proxy can inherit survivor selection because today's constituents may be represented in earlier years.

## EDA graph 2: rolling volatility

The chart uses a 20-session rolling standard deviation annualized by `sqrt(252)`. Annualization expresses a daily-scale dispersion in a familiar yearly scale; it does not imply that the measured volatility will persist for a full year. The curves can spike after unusually large returns and decline as those observations leave the rolling window.

Compare the *timing* and *relative pattern* of volatility across the sample tickers. Do not compare unadjusted share-price levels as a measure of risk. Different companies have different return distributions and event histories. A short window reacts quickly but is noisy; a long window is smoother but slower to reflect regime changes.

## EDA graph 3: momentum deciles and next-session up rate

`qcut` ranks observations by 20-session momentum and divides them into up to ten groups with roughly similar row counts. The bar height is the observed fraction of next-session up labels in that group. The horizontal line is the overall up-day fraction in the EDA sample.

If bars rise across deciles, that is a descriptive association in this dataset. If bars jump irregularly, the relationship may be weak, nonlinear, or dominated by sampling noise. Since many symbols share the same market dates, rows are not independent observations. The plot does not control for sector, market regime, trading costs, or repeated observations of the same symbol.

## Feature table and leakage review

| Feature or field | Information date | Role | Leakage check |
|---|---|---|---|
| `return_1d` | Current session close | Predictor | Uses price through date *t*. |
| `momentum_5d` | Current session close | Predictor | Uses current and earlier prices only. |
| `momentum_20d` | Current session close | Predictor | Uses current and earlier prices only. |
| `volatility_20d` | Current session close | Predictor | Rolling window ends at date *t*. |
| `volume_change_5d` | Current session close | Predictor | Difference uses volume at *t* and earlier. |
| `sector` | Current company reference file | Predictor | Current labels may not reflect historical classification. |
| `next_day_return` | Following observation | Label input | Never place in `X`. |
| `next_day_up` | Following observation | Target | Only used as `y`. |
| `target_date` | Following observation | Split control | Used to purge boundary crossing, not as a model feature. |

Always review the actual feature list printed at the end of the notebook. A column can be temporally close to the target and still leak future information if it is calculated after shifting or if a rolling window is centered. The rolling window used here is trailing, not centered.

## What to write after running the plots

Use observations rather than guesses. Record the actual last date and number of symbols, identify any visibly high-volatility periods, and describe whether the decile bars show a monotonic or irregular pattern. If no clear pattern appears, say so. A neutral finding is valid and often indicates why a baseline model is important.

Do not claim that momentum predicts returns based on the EDA bar chart alone. The baseline/ML notebook tests predictive performance on later dates. The EDA is for understanding distributions, coverage, and candidate features before training.

## Chart design choices and reproducibility

- **Date axis:** use actual trading dates, not a fabricated daily calendar. Weekends and market holidays have no bar.
- **Equal-weight proxy:** computes the mean of symbols available on each date. This keeps a changing row count visible but can change portfolio composition over time.
- **Rolling window:** `min_periods=20` ensures the volatility feature is not drawn from a shorter warm-up period.
- **Momentum bins:** duplicate quantile boundaries can occur when values repeat, so `duplicates='drop'` may produce fewer than ten bins. Read the actual labels rather than assuming every decile exists.
- **Sample stocks:** the five chart symbols are examples chosen for visual comparison. Their names do not define the full analysis sample.
- **Refresh version:** running the updated Kaggle download can alter the charts. Save a data version/date note with screenshots or reports intended for submission.

## Limitations that affect interpretation

Daily prices can contain split adjustments, dividend adjustments, ticker changes, and vendor corrections. If adjusted close is unavailable or incomplete, the pipeline uses close consistently, but total returns may omit cash dividends. The current-constituent sample also omits some delisted companies, which can bias historical performance upward. Sector metadata may be current rather than point-in-time. A single observed return is noisy, especially at a one-session horizon.

This EDA does not infer market capitalization or EBITDA. It explores the price/volume fields supplied by Kaggle and prepares a one-day direction label. If the research question changes back to fundamentals, a separate time-aligned filing source is needed.

# Extended chart interpretation and EDA worksheet

## A. Describe the sample before the pattern

Start every chart description with the population it actually displays. The EDA uses rows with enough prior observations to calculate features and a following return to create a target. The available sample is current Kaggle S&P 500 representation over its historical price file; it is not a verified point-in-time constituent panel. The count can therefore differ by date and by feature availability.

Record the row count, unique symbols, first and last feature dates, chosen price field, and positive-label rate printed by the notebook. These values explain what the chart covers and make later runs comparable. The file refresh may extend its last date or revise past prices.

## B. Read each chart in a consistent order

### Equal-weighted sample growth proxy

1. Read the title: it is a sample proxy, not an official index series.
2. Read the vertical scale: growth of one unit of capital under a daily equal-weighted average return calculation.
3. Identify drawdowns and recoveries by date; do not infer which stock caused them from this aggregate chart.
4. Note that the available names can change through time. A current-symbol backfill can produce survivorship bias.
5. Compare only like-for-like runs that use the same dataset version and price basis.

### Rolling realized volatility

1. Each line is one illustrative ticker, with a 20-session trailing window.
2. The annualized value is a rescaling of daily dispersion, not a guarantee of future annual risk.
3. Spikes can be caused by one or several large observations; verify with the underlying return series before explaining a specific event.
4. The window rolls forward, so volatility may drop when an extreme return leaves the sample.
5. Do not compare the dollar price level across tickers as a proxy for volatility.

### Momentum decile and next-day up frequency

1. The x-axis sorts observations by past 20-session momentum; the y-axis is the fraction of next observations with positive returns.
2. The horizontal line is the unconditional up frequency in the same feature-ready sample.
3. A monotone sequence would be consistent with a simple relationship in this sample, but a small decile gap can be noise.
4. The bars are not statistically independent because observations repeat by ticker and market date.
5. The ML validation set is the more appropriate place to judge whether the relationship carries forward.

## C. Questions for an EDA presentation

- Which chart answers a question about market growth, and which answers a question about risk?
- What does “equal weighted” mean for a date with fewer available symbols?
- Why does a rolling volatility line begin after the first 20 returns?
- Why can 20-day momentum be positive while the next-day return is negative?
- Which variable is the future label, and why is it not an input feature?
- What is lost when close is used instead of dividend-adjusted close?
- Why is a current constituent list not the same as historical S&P 500 membership?
- What additional test would you run before treating the momentum-bin pattern as predictive?

A strong explanation answers with the notebook's calculation and sample, not a general claim about how markets “usually” behave.

## D. Chart and feature formulas

**Simple return:** `P_t / P_(t-1) - 1`. A value of `0.02` represents a 2% increase from the preceding recorded session.

**k-session momentum:** `P_t / P_(t-k) - 1`. This uses the ratio of endpoint prices, not the sum of daily returns. It captures net movement over the lookback interval.

**Rolling realized volatility:** standard deviation of daily returns in the trailing 20 observations. The EDA multiplies by `sqrt(252)` to annualize under the conventional approximation of 252 sessions in a year.

**Log-volume change:** `log(1 + V_t) - log(1 + V_(t-5))`. This is a difference in log scale, not a percent change in dollar trading value. Volume is a count of shares and can be affected by split handling.

**Direction label:** `1` when the next recorded daily return is above zero. It does not encode return magnitude. A 0.01% positive return and a 10% positive return receive the same class.

## E. Common interpretation mistakes to avoid

- Calling the equal-weighted average a “market index” without the proxy qualification.
- Describing chart associations as causal effects.
- Saying high past momentum guarantees positive future returns.
- Treating all rows as independent while thousands of stocks share the same trading dates.
- Comparing close-based returns to total returns without noting the corporate-action basis.
- Assuming a price-history dataset contains EBITDA, market capitalization, or historical membership because the file name says S&P 500.
- Reporting one run's row count without the Kaggle refresh date.
- Filling missing chart observations with zeros, which would manufacture no-return days.

## F. Write an evidence-based chart caption

Use this template for each plot: “This chart shows ___ for ___ symbols over ___ to ___. The plotted measure is calculated as ___. The visible pattern is ___. This is descriptive because ___. A next step is ___.” Include only a pattern that is visible in the chart output from your run. If the chart is flat, noisy, or lacks a monotone relationship, say that plainly.

## G. Follow-up ideas supported by this price file

- Compare short and medium momentum horizons using only past prices.
- Compare rolling volatility estimates across 20-, 60-, and 120-session windows.
- Stratify the momentum relationship by sector, while noting sector membership is likely current metadata.
- Compare equal-weighted and value-weighted sample proxies only if a defensible market-cap series is available.
- Study drawdown, turnover, or return distributions by market regime.
- Evaluate all candidate ideas on chronological validation dates, then preserve a final test interval.

These extensions remain price-based. To study quarterly financial statement changes, join a separate SEC fundamental dataset with filing-date alignment and appropriate release lags.

## Practical interpretation worksheet

For each rendered chart, write one or two observations and one limitation:

| Chart | Observation to record | Limitation to mention |
|---|---|---|
| Equal-weight growth proxy | Direction, drawdown, or recovery dates visible in this run | Not the official cap-weighted S&P 500 index. |
| Rolling volatility | Dates when sample volatility rises or falls | A trailing estimate describes recent variation, not future risk. |
| Momentum deciles | Whether the bars rise, fall, or move irregularly relative to the overall rate | Descriptive groups do not control for time dependence or prove predictive value. |

The chart cells contain executable plotting code. Run them after loading the refreshed Parquet file and save the notebook so their actual figures appear between the code and interpretation notes. Do not paste a chart from an earlier Kaggle snapshot into a report about a later snapshot without recording the version difference.

If a graph looks empty, inspect `model_frame` row counts, selected symbols, date range, and missing values before changing the plotting code. Empty rolling windows at the beginning of a symbol history are expected because the feature needs 20 prior observations.

## A note on chart reproducibility

The charts are generated from the prepared Parquet snapshot. Their values can change when Kaggle updates older prices or appends new dates. Save the notebook after plotting, note the source snapshot, and use the same prepared file for any comparison of charts. If you rerun preprocessing with a newer snapshot, regenerate the feature panel instead of mixing old labels with new returns. The chart code deliberately avoids hard-coded numerical findings so the displayed interpretations remain tied to the data you actually ran.